# Day 89: Neyman Orthogonality

*Beginner to Ninja: Math for Causal Inference*

When a causal estimate depends on nuisance functions estimated with error (regressions of the outcome and treatment on many covariates), naive plug-in estimators inherit that error one-for-one. Orthogonal scores are insensitive to small nuisance errors, so the bias becomes a product of two errors. This is the key idea behind double machine learning.

**Learning outcomes**

- Define a moment condition with nuisance parameters and the Neyman orthogonality condition.
- Show that the partialling-out score in the partially linear model is orthogonal and the naive score is not.
- Prove that the AIPW score is orthogonal and that its bias is a product of nuisance errors.
- Demonstrate first-order versus second-order bias by perturbing nuisances.

## Theory

### Moments with nuisances

Many estimators solve $E[\psi(W; \theta, \eta)] = 0$, where $\theta$ is the target and $\eta$ a **nuisance** (regression functions, propensity scores). In practice we plug in an estimate $\hat\eta$, often from regularised or machine-learning fits that converge slowly and are biased (shrinkage).

### Orthogonality

The score is **Neyman orthogonal** if small changes in the nuisance do not change the moment, to first order:

$$
\frac{\partial}{\partial r}E\big[\psi(W; \theta_0, \eta_0 + r(\eta - \eta_0))\big]\Big|_{r = 0} = 0 \quad \text{for all directions } \eta - \eta_0.
$$

Then $\hat\theta - \theta_0$ depends on $\hat\eta - \eta_0$ only at second order, typically through products such as $\|\hat m - m\|\cdot\|\hat\ell - \ell\|$. If each nuisance converges faster than $n^{-1/4}$, the product is $o(n^{-1/2})$ and $\hat\theta$ behaves as if the nuisances were known.

### The partially linear model

$$
Y = \theta D + g(X) + \varepsilon, \qquad D = m(X) + v, \qquad E[\varepsilon \mid X, D] = 0,\; E[v \mid X] = 0.
$$

- **Naive score:** $\psi = (Y - \theta D - g(X))D$. An error in $\hat g$ shifts $\hat\theta$ by about $E[D(g - \hat g)]/E[D^2]$: first-order bias, because $D$ is correlated with $X$.
- **Partialling-out (Robinson) score:** with $\ell(X) = E[Y \mid X]$,

$$
\psi = \big(Y - \ell(X) - \theta(D - m(X))\big)\big(D - m(X)\big),
$$

which gives the residual-on-residual estimator $\hat\theta = \sum\tilde Y\tilde D/\sum\tilde D^2$ (FWL with flexible first stages). It is orthogonal in $(\ell, m)$.

### The AIPW score

For the ATE, $\psi = \mu_1(X) - \mu_0(X) + \frac{D(Y - \mu_1(X))}{e(X)} - \frac{(1 - D)(Y - \mu_0(X))}{1 - e(X)} - \theta$ (Day 87) is orthogonal in $(\mu_0, \mu_1, e)$. Its bias is

$$
E\Big[\frac{(e(X) - \hat e(X))(\mu_1(X) - \hat\mu_1(X))}{\hat e(X)}\Big] + (\text{analogous control term}),
$$

a product: the source of double robustness and of fast convergence with ML nuisances.

### Constructing orthogonal scores

Starting from any identifying score, subtract its projection on the nuisance tangent space; for regression problems this amounts to residualising. Efficient influence functions (Days 87–88) are orthogonal by construction.

## Part 1 · Warm-up

### Exercise 1 · Spot the nuisances

Name the target and the nuisances in (a) the partialling-out score; (b) the AIPW score.

<details><summary>Show solution</summary>

**Step 1.** (a) Target $\theta$; nuisances $\ell(X) = E[Y \mid X]$ and $m(X) = E[D \mid X]$.

**Step 2.** (b) Target the ATE; nuisances $\mu_0, \mu_1, e$.

</details>

*Your work:*

### Exercise 2 · First-order bias of the naive score

Suppose $\hat g = g + 0.1$ (a constant error) and $E[D] = 0.5$, $E[D^2] = 0.5$ (binary treatment). What is the bias of the naive estimator?

<details><summary>Show solution</summary>

**Step 1.** Bias $\approx E[D(g - \hat g)]/E[D^2] = -0.1 \times 0.5/0.5 = -0.1$.

**Answer:** the full nuisance error passes into $\hat\theta$.

</details>

*Your work:*

### Exercise 3 · The same error, orthogonal score

Now $\hat\ell = \ell + 0.1$ with $\hat m = m$. What is the bias of the partialling-out estimator?

<details><summary>Show solution</summary>

**Step 1.** $\tilde Y$ shifts by $-0.1$; the estimator changes by $-0.1\,E[\tilde D]/E[\tilde D^2]$.

**Step 2.** $E[\tilde D] = E[D - m(X)] = 0$.

**Answer:** no bias: a constant error in $\ell$ is orthogonal to the treatment residual.

</details>

*Your work:*

### Exercise 4 · Rates

Each nuisance estimate has error of order $n^{-0.3}$. Is the product of order $o(n^{-1/2})$?

<details><summary>Show solution</summary>

**Step 1.** Product: $n^{-0.6}$, and $0.6 > 0.5$.

**Answer:** yes; with errors of order $n^{-0.2}$ each the product $n^{-0.4}$ would not be.

</details>

*Your work:*

### Exercise 5 · Why not just OLS with all controls?

With 500 covariates and 1,000 observations, why is OLS of $Y$ on $D$ and all $X$ unattractive, and why does naive ML not fix it?

<details><summary>Show solution</summary>

**Step 1.** OLS with nearly as many regressors as observations is very noisy (or impossible beyond $n$).

**Step 2.** Regularised ML fits of $g$ are biased (shrinkage); with the naive score that bias passes into $\hat\theta$ at first order.

**Answer:** orthogonal scores plus ML first stages (with cross-fitting, Day 90) solve both problems.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · The partialling-out score is orthogonal

Let $\psi(W; \theta, \ell, m) = (Y - \ell - \theta(D - m))(D - m)$. Show the Gâteaux derivatives in directions $h_\ell(X)$ and $h_m(X)$ vanish at the truth.

<details><summary>Show solution</summary>

**Step 1.** Direction $\ell$: $\frac{\partial}{\partial r}E[\psi]$ $= -E[h_\ell(X)(D - m(X))] = -E[h_\ell(X)E[v \mid X]] = 0$.

**Step 2.** Direction $m$: derivative $= E[\theta h_m(X)(D - m)] - E[(Y - \ell - \theta(D - m))h_m(X)]$.

**Step 3.** First term: $\theta E[h_mE[v \mid X]] = 0$. Second: $Y - \ell - \theta v = \varepsilon + (g + \theta m - \ell) = \varepsilon$ (since $\ell = \theta m + g$), and $E[\varepsilon h_m(X)] = 0$. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · The naive score is not orthogonal

Show that for $\psi = (Y - \theta D - g(X))D$, the derivative in direction $h_g$ is $-E[D h_g(X)]$, generally non-zero.

<details><summary>Show solution</summary>

**Step 1.** $\frac{\partial}{\partial r}E[(Y - \theta D - g - rh_g)D] = -E[h_g(X)D] = -E[h_g(X)m(X)]$.

**Step 2.** Non-zero whenever $m(X)$ (the part of treatment predicted by covariates) correlates with the error direction. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · AIPW is orthogonal in μ₁

Show the derivative of $E[\psi_{AIPW}]$ in direction $h(X)$ for $\mu_1$ is zero at the truth.

<details><summary>Show solution</summary>

**Step 1.** $\mu_1$ enters as $\mu_1 - D\mu_1/e$, so the derivative is $E[h(X)(1 - D/e(X))]$.

**Step 2.** Conditioning on $X$: $E[1 - D/e(X) \mid X] = 1 - e(X)/e(X) = 0$. $\blacksquare$ The same holds for $\mu_0$ and (using $E[Y - \mu_1 \mid D = 1, X] = 0$) for $e$.

</details>

*Your work:*

### Exercise 9 · The product form of AIPW's bias

Show that with nuisances $\hat\mu_1, \hat e$ in place of the truth (control part ignored), $E\big[\hat\mu_1 + D(Y - \hat\mu_1)/\hat e\big] - E[\mu_1] = E\big[(e - \hat e)(\mu_1 - \hat\mu_1)/\hat e\big]$.

<details><summary>Show solution</summary>

**Step 1.** Condition on $X$: $E[D(Y - \hat\mu_1) \mid X] = e(\mu_1 - \hat\mu_1)$.

**Step 2.** So the left side is $E\big[\hat\mu_1 + e(\mu_1 - \hat\mu_1)/\hat e - \mu_1\big] = E\big[(\mu_1 - \hat\mu_1)(e/\hat e - 1)\big]$.

**Step 3.** $e/\hat e - 1 = (e - \hat e)/\hat e$. $\blacksquare$ Zero if either nuisance is right (double robustness); small if both are nearly right.

</details>

*Your work:*

### Exercise 10 · From product bias to √n-consistency

Assume $\hat e$ is bounded away from 0 and $\|\hat e - e\|_2\,\|\hat\mu_1 - \mu_1\|_2 = o_p(n^{-1/2})$. Use Cauchy–Schwarz to show the bias in Exercise 9 is $o_p(n^{-1/2})$.

<details><summary>Show solution</summary>

**Step 1.** $\big|E[(e - \hat e)(\mu_1 - \hat\mu_1)/\hat e]\big| \le \frac1c E\big[|e - \hat e|\,|\mu_1 - \hat\mu_1|\big]$ with $\hat e \ge c$.

**Step 2.** Cauchy–Schwarz: $\le \frac1c\|e - \hat e\|_2\|\mu_1 - \hat\mu_1\|_2 = o_p(n^{-1/2})$. $\blacksquare$ (Cross-fitting, Day 90, handles the dependence between $\hat\eta$ and the sample.)

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · Pension eligibility and wealth with many controls

In a simulation in the spirit of the 401(k) studies (50 covariates, 1,000 households, true effect 0.5), the nuisances are fitted by heavily penalised ridge regressions. Compare the naive estimator (penalising controls in a joint regression) with partialling-out.

<details><summary>Show solution</summary>

**Step 1.** Naive: about 0.89, badly biased: shrinking the control coefficients leaves confounding that loads onto $D$.

**Step 2.** Partialling-out: about 0.59, much closer: shrinkage errors in $\hat\ell$ and $\hat m$ enter only through their product.

**Answer:** orthogonality cuts the bias by three-quarters with the same nuisance fits; better (less biased) nuisances or cross-fitting reduce it further.

</details>

*Your work:*

### Exercise 12 · Air pollution and hospital admissions

A study regresses daily asthma admissions on PM2.5, controlling flexibly for weather (temperature, humidity, wind, with lags and interactions: hundreds of terms). Describe an orthogonal estimator.

<details><summary>Show solution</summary>

**Step 1.** Partially linear model: admissions $= \theta\,\text{PM2.5} + g(\text{weather, calendar}) + \varepsilon$.

**Step 2.** Fit $\hat\ell$ (admissions on weather) and $\hat m$ (PM2.5 on weather) with flexible ML, using cross-fitting.

**Step 3.** Regress residualised admissions on residualised PM2.5.

</details>

*Your work:*

### Exercise 13 · First- versus second-order bias

In the code below, nuisance functions are perturbed by $r \times$ (a fixed error) for $r = 0.1, 0.2, 0.4$. How does the bias of the naive and orthogonal estimators scale with $r$?

<details><summary>Show solution</summary>

**Step 1.** Naive: biases $-0.018, -0.037, -0.075$: proportional to $r$ (doubling $r$ doubles it).

**Step 2.** Orthogonal: biases $+0.004, +0.013, +0.048$: each doubling multiplies the bias by about 3.5, close to the factor 4 of an $r^2$ law, from a much smaller base.

**Answer:** small nuisance errors hurt the naive estimator linearly but the orthogonal one only quadratically, which is why imperfect ML first stages are tolerable with orthogonal scores.

</details>

*Your work:*

### Exercise 14 · Both nuisances wrong

In an AIPW analysis of a smoking-cessation drug, the propensity model omits a variable and the outcome model omits another. Is the estimate consistent?

<details><summary>Show solution</summary>

**Step 1.** By Exercise 9, bias $= E[(e - \hat e)(\mu - \hat\mu)/\hat e]$, non-zero if both errors are non-zero and correlated.

**Answer:** generally not, but if each model is only mildly wrong, the bias (a product) is small; that is why flexible fits for both nuisances matter.

</details>

*Your work:*

### Exercise 15 · Explaining orthogonality

Explain to a policy audience, without equations, why "residual-on-residual" regression protects the effect estimate from imperfect control-variable models.

<details><summary>Show solution</summary>

**Step 1.** We first remove from both the outcome and the treatment everything the controls predict.

**Step 2.** The effect is then estimated from the leftover variation in treatment, which by construction is unrelated to the controls.

**Answer:** small mistakes in the control models shift the leftovers in ways that are mostly unrelated to each other, so they cancel to first order rather than adding up.

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np

rng = np.random.default_rng(89)
n, p = 1000, 50
bg = np.r_[np.linspace(1, .2, 10), np.zeros(p - 10)]; bm = np.r_[np.linspace(.8, .1, 10), np.zeros(p - 10)]
naive, orth = [], []
for _ in range(500):
    X = rng.normal(size=(n, p)); D = X @ bm + rng.normal(size=n); Y = .5 * D + X @ bg + rng.normal(size=n)
    Z = np.column_stack([D, X]); P = np.eye(p + 1) * 200; P[0, 0] = 0
    naive.append(np.linalg.solve(Z.T @ Z + P, Z.T @ Y)[0])                 # ridge on controls, D unpenalised
    ridge = lambda y: X @ np.linalg.solve(X.T @ X + 200 * np.eye(p), X.T @ y)
    rY, rD = Y - ridge(Y), D - ridge(D)
    orth.append(rD @ rY / (rD @ rD))
print(f"naive {np.mean(naive):.3f} (SD {np.std(naive):.3f})   partialling-out {np.mean(orth):.3f} (SD {np.std(orth):.3f})   truth 0.5")

# Exercise 13: perturb known nuisances by r * error and compare biases (large n, one draw)
N = 2_000_000; x = rng.normal(size=N); m = np.tanh(x); g = x ** 2
D = m + rng.normal(size=N); Y = .5 * D + g + rng.normal(size=N); ell = .5 * m + g
for r in (.1, .2, .4):
    gh = g + r * np.sin(2 * x) + r                     # error direction for g / ell
    mh = m + r * np.cos(x)                              # error direction for m
    th_naive = ((Y - gh) * D).mean() / (D * D).mean()
    lh = ell + r * np.sin(2 * x) + r
    rD = D - mh; th_orth = ((Y - lh) * rD).mean() / (rD * rD).mean()
    print(f"r={r}: naive bias {th_naive - .5:+.4f}   orthogonal bias {th_orth - .5:+.4f}")

## Key takeaways

- A score is Neyman orthogonal when small nuisance errors leave its expectation unchanged to first order.
- Partialling-out (residual-on-residual) and AIPW scores are orthogonal; naive plug-in scores are not.
- With orthogonal scores, bias is a product of nuisance errors, so ML nuisances converging faster than n^(−1/4) leave valid √n inference.